### 이진 분류
: banknote authentification dataset : 진짜 지폐와 가짜 지폐를 판별

- 데이터 : 1372개의 instance
- 정답(variance): 0-> 진짜, 1 -> 가짜를 의미
- 특징(입력특징): 4개의 특징
    - variance(분산): 이미지를 wavelit변환해서 얻은 값들이 퍼져있는 정도
    - skewness(왜도): 비대칭성
    - curtosis(첨도): 극단성
    - entropy(엔트로피): 불확실성(밝기)
        - 낮으면 특정 값이나 패턴에 집중되어 있음(불확실성 낮음)
        - 높다면 다양한 값이 나타나고 분포가 복잡해지는 경향(불확실성 높음)

- 4개의 특징에 대한 입력값으로 위조지폐를 찾아내는 모델을 개발

In [11]:
import pandas as  pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import StandardScaler, MinMaxScaler

import random

#### Step 1. 데이터 불러오기 및 전처리

In [ ]:
columns = ['variance', 'skewness', 'curtosis', 'entropy', 'class']
df = pd.read_csv('data_banknote_authentication.txt', names= columns)
print(df.head())
print('데이터의 크기:', df.shape)                 # 1372개의 instance
print('클래스별 개수:')
print(df['class'].value_counts().sort_index())  # 가짜: 610, 진짜: 762

   variance  skewness  curtosis  entropy  class
0   3.62160    8.6661   -2.8073 -0.44699      0
1   4.54590    8.1674   -2.4586 -1.46210      0
2   3.86600   -2.6383    1.9242  0.10645      0
3   3.45660    9.5228   -4.0112 -3.59440      0
4   0.32924   -4.4552    4.5718 -0.98880      0
데이터의 크기: (1372, 5)
클래스별 개수:
class
0    762
1    610
Name: count, dtype: int64


In [ ]:
# Input, Target 데이터 분리
# target = 'class'
# x = df.drop(target, axis = 1)
# y = df.loc[:, target]
x = torch.tensor(df[['variance','skewness','curtosis','entropy']].values, dtype= torch.float32)
y = torch.tensor(df[['class']].values, dtype = torch.float32).reshape(-1,1)

# 데이터셋 생성
torch.manual_seed(42)   # 은하수를 여행하는 히치하이커를 위한 안내서
# x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3 ,randomseed=20)
train_indices = []
test_indices = []

# # train_test_split을 하드코딩
# for class_value in [0,1]:
#     class_indices = []

#     for i in range(len(y)):
#         if y[i].item() == class_value:
#             class_indices.append(i)

#     # 순서를 무작위로 섞기
#     random.shuffle(class_indices)

#     # 학습 80%, 테스트 20% 분리
#     split = int(len(class_indices) * 0.8)

#     # extend() : list에 다른 리스트의 요소를 집어넣는 메소드
#     train_indices.extend(class_indices[:split])
#     test_indices.extend(class_indices[split:])

# print(len(test_indices))
# print(len(train_indices))

# 함수
for class_value in [0,1] :
    # where() : 조건을 만족하는 데이터 찾기. 결과는 튜플로 반환
    class_indices = torch.where(y.squeeze() == class_value)[0]

    # randperm() : 0부터 지정한 숫자까지의 정수를 무작위로 섞어서 반환
    class_indices = class_indices[torch.randperm(len(class_indices))]

    # 데이터 나누기
    train_indices.extend(class_indices[:split])
    test_indices.extend(class_indices[split:])

275
1097
